In [ ]:
# | default_exp schema_extractor

In [ ]:
# | export
from sqlmodel import Session, select
from seootter.article import get_article_by_path, insert_article, Article
from seootter.sqlite_db import get_session
from seootter.models import Website
from seootter.content_mapper import map_all_urls_to_files, fetch_sitemap_urls

In [ ]:
# | hide
#| eval: false
from dotenv import load_dotenv
import os

load_dotenv()
# BASE_PATH = os.environ["SEEOOTTER_BASE_PATH"]
BASE_PATH = os.environ["AWAZLY_PATH"]
BASE_PATH = os.environ["SHELID_PATH"]


In [ ]:
#| export 
from seootter.gsc.queries import get_top_queries
from seootter.article import get_articles_by_website
from seootter.gsc_client import get_date_range

# Schema Extractor
> Extract and parse JSON-LD schema markup from web pages.

## FAQ Extractor (Rule-Based, Arabic + English)

In [ ]:
# | export
import re

_ARABIC_RE = re.compile(r"[\u0600-\u06FF]")

# English question word patterns
_EN_Q_WORDS = re.compile(
    r"^(what|who|why|when|where|which|how|can|is|are|do|does|did|will|should|"
    r"could|would|may|might|shall)\b", re.IGNORECASE)

# Arabic question word patterns — requires space after particle to avoid matching nouns
_AR_Q_WORDS = re.compile(
    r"^("
    r"ما (هو|هي|هم|هي|الح|ال|يع|تع|يج|ه[ا-ي])|"
    r"ماذا |من أين |من اين |أين |اين |كيف |كيفية |كيفيه |"
    r"هل |لماذا |كم |بماذا |لمن |متى |أي |من "
    r")")


def _is_arabic(text: str  # Text to check
               ) -> bool:
    "Return True if more than 30% of characters are Arabic."
    return len(_ARABIC_RE.findall(text)) > len(text) * 0.3


def _is_question(query: str  # Search query
                 ) -> bool:
    "Return True if the query is a question in Arabic or English."
    q = query.strip()
    if q.endswith("?") or q.endswith("؟"): return True
    return bool(_AR_Q_WORDS.match(q) if _is_arabic(q) else _EN_Q_WORDS.match(q))


def extract_faq_queries(rows: list[dict] | list[str]  # GSC query dicts or plain strings
                        ) -> list[str]:
    "Filter GSC queries that are questions in Arabic or English."
    queries = (r["query"] if isinstance(r, dict) else r for r in rows)
    return list(filter(_is_question, queries))



In [ ]:
#| hide
#| eval: false

path = "https://shelid.com/"
website_name = "https://shelid.com/"

start, end = get_date_range("last_days", days=90 * 8)
print(start, end)
with get_session() as session:
    articles = get_articles_by_website(session, website_id=2)
    for article in articles:
        page_path = article.url.removeprefix(website_name)
        top_query = get_top_queries(session=session, site_url="sc-domain:shelid.com", start_date=start, end_date=end,
                                    page_path=page_path, limit=1000, sort_by="impressions")
        if extract_faq_queries(top_query) == []:
            continue
        print(extract_faq_queries(top_query))
        print(article.file_path)


In [ ]:
#| hide
#| eval: false
should_match = [
    # Arabic questions
    "ما هو البورديوم", "ما هي خامة البورديوم", "ما هو grc في البناء",
    "كيف انظف الكنب المخمل من البقع", "كيفية تنظيف الكنب المخمل",
    "هل اسطوانة الغاز تنفجر", "هل الايبوكسي افضل ام السيراميك",
    "كم سعر عزل الفوم", "كم وزن اسطوانة الغاز",
    "اين يباع جهاز كشف تسرب المياه", "من اين استطيع شراء انبوبة غاز جديده",
    "لماذا يهمنا سرعة الصفحة",
    # English questions
    "how to improve page speed", "what is a backlink", "is fiber gas cylinder safe",
]

should_not_match = [
    "ماكينة غسيل الكنب", "مادة grc", "مادة الايبوكسي", "منظف كنب مخمل",
    "مانع تسرب السوائل ساكو", "ماسورة غاز", "ماصورة غاز",
    "منازل جده", "مادة بناء", "منظم غاز", "best seo tools", "digital marketing",
]

print("✓ Should match (questions):")
for q in should_match:
    result = _is_question(q)
    mark = "✓" if result else "✗ MISSED"
    print(f"  {mark}  {q}")

print("\n✗ Should NOT match (not questions):")
for q in should_not_match:
    result = _is_question(q)
    mark = "✗ FALSE POSITIVE" if result else "✓"
    print(f"  {mark}  {q}")
